# 01 · Data Preparation

## Objetivo

El objetivo de este notebook es preparar los datos originales para su posterior análisis.

Durante esta fase se cargan los datos, se limpian, se transforman y se organizan en un modelo relacional compuesto por las tablas **customers**, **products** y **orders**.

El resultado final será un conjunto de datos consistente y estructurado, listo para ser utilizado en el análisis con SQL, Python y Power BI.

In [99]:
# ==========================
# Importacion de librerias
# ==========================

import pandas as pd
from pathlib import Path

In [100]:
# ==========================
# Carga de datos
# ==========================

DATA_PATH = Path("../data/raw/Electronic_sales_Sep2023-Sep2024.csv")

df = pd.read_csv(DATA_PATH)

In [101]:
# ===============================
# Inspeccion inicial de los datos
# ===============================

df.head()

,Customer ID,Age,Gender,Loyalty Member,Product Type,SKU,Rating,Order Status,Payment Method,Total Price,Unit Price,Quantity,Purchase Date,Shipping Type,Add-ons Purchased,Add-on Total
0,1000,53,Male,No,Smartphone,SKU1004,2,Cancelled,Credit Card,5538.33,791.19,7,2024-03-20,Standard,"Accessory,Accessory,Accessory",40.21
1,1000,53,Male,No,Tablet,SKU1002,3,Completed,Paypal,741.09,247.03,3,2024-04-20,Overnight,Impulse Item,26.09
2,1002,41,Male,No,Laptop,SKU1005,3,Completed,Credit Card,1855.84,463.96,4,2023-10-17,Express,NaN,0.00
3,1002,41,Male,Yes,Smartphone,SKU1004,2,Completed,Cash,3164.76,791.19,4,2024-08-09,Overnight,"Impulse Item,Impulse Item",60.16
4,1003,75,Male,Yes,Smartphone,SKU1001,5,Completed,Cash,41.50,20.75,2,2024-05-21,Express,Accessory,35.56


In [102]:
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 20000 entries, 0 to 19999
Data columns (total 16 columns):
 #   Column             Non-Null Count  Dtype  
---  ------             --------------  -----  
 0   Customer ID        20000 non-null  int64  
 1   Age                20000 non-null  int64  
 2   Gender             19999 non-null  str    
 3   Loyalty Member     20000 non-null  str    
 4   Product Type       20000 non-null  str    
 5   SKU                20000 non-null  str    
 6   Rating             20000 non-null  int64  
 7   Order Status       20000 non-null  str    
 8   Payment Method     20000 non-null  str    
 9   Total Price        20000 non-null  float64
 10  Unit Price         20000 non-null  float64
 11  Quantity           20000 non-null  int64  
 12  Purchase Date      20000 non-null  str    
 13  Shipping Type      20000 non-null  str    
 14  Add-ons Purchased  15132 non-null  str    
 15  Add-on Total       20000 non-null  float64
dtypes: float64(3), int64(4), str(9)
m

In [103]:
df.shape

(20000, 16)

In [104]:
df.columns

Index(['Customer ID', 'Age', 'Gender', 'Loyalty Member', 'Product Type', 'SKU',
       'Rating', 'Order Status', 'Payment Method', 'Total Price', 'Unit Price',
       'Quantity', 'Purchase Date', 'Shipping Type', 'Add-ons Purchased',
       'Add-on Total'],
      dtype='str')

In [105]:
df[df["Customer ID"] == 1002]

,Customer ID,Age,Gender,Loyalty Member,Product Type,SKU,Rating,Order Status,Payment Method,Total Price,Unit Price,Quantity,Purchase Date,Shipping Type,Add-ons Purchased,Add-on Total
2,1002,41,Male,No,Laptop,SKU1005,3,Completed,Credit Card,1855.84,463.96,4,2023-10-17,Express,NaN,0.00
3,1002,41,Male,Yes,Smartphone,SKU1004,2,Completed,Cash,3164.76,791.19,4,2024-08-09,Overnight,"Impulse Item,Impulse Item",60.16


# Preparación de los datos

Antes de construir las tablas del modelo relacional, es necesario preparar el conjunto de datos.

La columna **Purchase Date** se convertirá al tipo de dato **datetime** para poder identificar la transacción más reciente de cada cliente.

In [106]:
df["Purchase Date"] = pd.to_datetime(
    df["Purchase Date"],
    format="%Y-%m-%d"
)

In [107]:
df = df.sort_values(by="Purchase Date", ascending=False)

In [108]:
df = df.drop_duplicates(subset="Customer ID", keep="first")

In [109]:
# ==================================
# Construccion de la tabla customers
# ==================================

customers = df[
    [
        "Customer ID",
        "Age",
        "Gender",
        "Loyalty Member"
    ]
]

customers = customers.reset_index(drop=True)

customers.head()

,Customer ID,Age,Gender,Loyalty Member
0,19033,47,Female,No
1,1402,77,Male,No
2,14264,36,Male,No
3,9976,36,Female,No
4,19471,54,Female,No


## Interpretación

El conjunto de datos contiene **12.136 clientes únicos** y **20.000 transacciones de compra**.

Esto indica que muchos clientes realizaron más de una compra durante el período analizado.

De media, cada cliente realizó aproximadamente **1,65 compras**, lo que sugiere que el análisis de la fidelización y la recurrencia de compra puede aportar información de gran valor para el negocio.

In [110]:
# ==================================
# Eliminar el registro inconsistente
# ==================================

df = df[
    ~(
        (df["SKU"] == "SKU1005") &
        (df["Product Type"] == "Smartphone")
    )
]

In [111]:
# ====================================
# Comprobar la consistencia de los SKU
# ====================================

df.groupby("SKU")["Product Type"].nunique()

SKU
HDP456     1
LTP123     1
SKU1001    1
SKU1002    1
SKU1003    1
SKU1004    1
SKU1005    1
SMP234     1
SWT567     1
TBL345     1
Name: Product Type, dtype: int64

In [112]:
df[df["SKU"] == "SKU1005"]["Product Type"].value_counts()

Product Type
Laptop    1228
Name: count, dtype: int64

In [113]:
df[df["SKU"] == "SKU1005"]["Product Type"].unique()

<StringArray>
['Laptop']
Length: 1, dtype: str

In [114]:
df[
    (df["SKU"] == "SKU1005") &
    (df["Product Type"] == "Smartphone")]

,Customer ID,Age,Gender,Loyalty Member,Product Type,SKU,Rating,Order Status,Payment Method,Total Price,Unit Price,Quantity,Purchase Date,Shipping Type,Add-ons Purchased,Add-on Total


# Validación y calidad de los datos

Durante la validación de la tabla **products** se detectó una inconsistencia en el conjunto de datos.

El **SKU1005** aparecía asociado a dos tipos de producto diferentes. Tras analizar los registros, se comprobó que el valor inconsistente también presentaba un precio unitario distinto al resto de registros con ese SKU, lo que indica que el problema probablemente se encuentra en el identificador del producto y no únicamente en el tipo de producto.

Dado que no existe evidencia suficiente para corregir el SKU de forma fiable, se decidió excluir este registro del proceso de construcción de la tabla **products**, manteniendo la integridad del modelo de datos.

# Construcción de la tabla de productos

Una vez validada la consistencia de los datos, se construye la tabla **products**.

Cada producto se identifica mediante un **product_id** (clave sustituta), además de conservar el **SKU** y el **tipo de producto** como atributos descriptivos.

In [115]:
products = (
    df[["SKU", "Product Type"]]
    .drop_duplicates()
    .sort_values("SKU")
    .reset_index(drop=True)
)

In [116]:
# ==================================
# Crear la clave primaria product_id
# ==================================

products.insert(
    0,
    "product_id",
    range(1, len(products) + 1)
)

products

,product_id,SKU,Product Type
0,1,HDP456,Headphones
1,2,LTP123,Laptop
2,3,SKU1001,Smartphone
3,4,SKU1002,Tablet
4,5,SKU1003,Smartwatch
5,6,SKU1004,Smartphone
6,7,SKU1005,Laptop
7,8,SMP234,Smartphone
8,9,SWT567,Smartwatch
9,10,TBL345,Tablet


## Interpretación

Se ha construido correctamente la dimensión **products**, compuesta por **10 productos únicos**.

Durante el proceso de validación se detectó una inconsistencia en uno de los registros, que fue excluido para garantizar la integridad de la información.

Cada producto dispone de un identificador interno (`product_id`), que actuará como clave primaria en la base de datos y facilitará la relación con la tabla de pedidos.

# Construcción de la tabla de pedidos

Una vez creadas las dimensiones **customers** y **products**, el siguiente paso consiste en construir la tabla **orders**.

Para ello, primero es necesario relacionar cada pedido con el producto correspondiente mediante el identificador `product_id`, que actuará como clave foránea en el modelo relacional.

In [117]:
# ==================================
# Relacionar pedidos con productos
# ==================================

orders_preview = df.merge(
    products,
    on="SKU",
    how="left"
)

orders_preview.head()

,Customer ID,Age,Gender,Loyalty Member,Product Type_x,SKU,Rating,Order Status,Payment Method,Total Price,Unit Price,Quantity,Purchase Date,Shipping Type,Add-ons Purchased,Add-on Total,product_id,Product Type_y
0,19033,47,Female,No,Tablet,TBL345,4,Completed,Bank Transfer,4718.46,786.41,6,2024-09-23,Standard,"Impulse Item, Impulse Item",187.94,10,Tablet
1,1402,77,Male,No,Smartphone,SKU1004,2,Cancelled,Paypal,3955.95,791.19,5,2024-09-23,Overnight,"Accessory,Accessory,Extended Warranty",99.39,6,Smartphone
2,14264,36,Male,No,Smartphone,SMP234,2,Completed,Credit Card,11396.80,1139.68,10,2024-09-23,Standard,"Impulse Item, Impulse Item",163.65,8,Smartphone
3,9976,36,Female,No,Tablet,SKU1002,3,Completed,Paypal,2223.27,247.03,9,2024-09-23,Overnight,Accessory,8.97,4,Tablet
4,19471,54,Female,No,Tablet,TBL345,3,Cancelled,Credit Card,3145.64,786.41,4,2024-09-23,Same Day,"Extended Warranty, Extended Warranty",145.55,10,Tablet


In [118]:
# ==================================
# Construcción de la tabla orders
# ==================================

orders = (
    orders_preview[
        [
            "Customer ID",
            "product_id",
            "Rating",
            "Order Status",
            "Payment Method",
            "Total Price",
            "Unit Price",
            "Quantity",
            "Purchase Date",
            "Shipping Type",
            "Add-ons Purchased",
            "Add-on Total"
        ]
    ]
    .reset_index(drop=True)
)

orders.head()

,Customer ID,product_id,Rating,Order Status,Payment Method,Total Price,Unit Price,Quantity,Purchase Date,Shipping Type,Add-ons Purchased,Add-on Total
0,19033,10,4,Completed,Bank Transfer,4718.46,786.41,6,2024-09-23,Standard,"Impulse Item, Impulse Item",187.94
1,1402,6,2,Cancelled,Paypal,3955.95,791.19,5,2024-09-23,Overnight,"Accessory,Accessory,Extended Warranty",99.39
2,14264,8,2,Completed,Credit Card,11396.80,1139.68,10,2024-09-23,Standard,"Impulse Item, Impulse Item",163.65
3,9976,4,3,Completed,Paypal,2223.27,247.03,9,2024-09-23,Overnight,Accessory,8.97
4,19471,10,3,Cancelled,Credit Card,3145.64,786.41,4,2024-09-23,Same Day,"Extended Warranty, Extended Warranty",145.55


In [119]:
# ==================================
# Crear la clave primaria order_id
# ==================================

orders.insert(
    0,
    "order_id",
    range(1, len(orders) + 1)
)

orders.head()

,order_id,Customer ID,product_id,Rating,Order Status,Payment Method,Total Price,Unit Price,Quantity,Purchase Date,Shipping Type,Add-ons Purchased,Add-on Total
0,1,19033,10,4,Completed,Bank Transfer,4718.46,786.41,6,2024-09-23,Standard,"Impulse Item, Impulse Item",187.94
1,2,1402,6,2,Cancelled,Paypal,3955.95,791.19,5,2024-09-23,Overnight,"Accessory,Accessory,Extended Warranty",99.39
2,3,14264,8,2,Completed,Credit Card,11396.80,1139.68,10,2024-09-23,Standard,"Impulse Item, Impulse Item",163.65
3,4,9976,4,3,Completed,Paypal,2223.27,247.03,9,2024-09-23,Overnight,Accessory,8.97
4,5,19471,10,3,Cancelled,Credit Card,3145.64,786.41,4,2024-09-23,Same Day,"Extended Warranty, Extended Warranty",145.55


In [120]:
df.shape

(12135, 16)

In [121]:
df.info()

<class 'pandas.DataFrame'>
Index: 12135 entries, 18920 to 5937
Data columns (total 16 columns):
 #   Column             Non-Null Count  Dtype         
---  ------             --------------  -----         
 0   Customer ID        12135 non-null  int64         
 1   Age                12135 non-null  int64         
 2   Gender             12134 non-null  str           
 3   Loyalty Member     12135 non-null  str           
 4   Product Type       12135 non-null  str           
 5   SKU                12135 non-null  str           
 6   Rating             12135 non-null  int64         
 7   Order Status       12135 non-null  str           
 8   Payment Method     12135 non-null  str           
 9   Total Price        12135 non-null  float64       
 10  Unit Price         12135 non-null  float64       
 11  Quantity           12135 non-null  int64         
 12  Purchase Date      12135 non-null  datetime64[us]
 13  Shipping Type      12135 non-null  str           
 14  Add-ons Purchased  

In [122]:
df.head()

,Customer ID,Age,Gender,Loyalty Member,Product Type,SKU,Rating,Order Status,Payment Method,Total Price,Unit Price,Quantity,Purchase Date,Shipping Type,Add-ons Purchased,Add-on Total
18920,19033,47,Female,No,Tablet,TBL345,4,Completed,Bank Transfer,4718.46,786.41,6,2024-09-23,Standard,"Impulse Item, Impulse Item",187.94
451,1402,77,Male,No,Smartphone,SKU1004,2,Cancelled,Paypal,3955.95,791.19,5,2024-09-23,Overnight,"Accessory,Accessory,Extended Warranty",99.39
13678,14264,36,Male,No,Smartphone,SMP234,2,Completed,Credit Card,11396.80,1139.68,10,2024-09-23,Standard,"Impulse Item, Impulse Item",163.65
9978,9976,36,Female,No,Tablet,SKU1002,3,Completed,Paypal,2223.27,247.03,9,2024-09-23,Overnight,Accessory,8.97
19400,19471,54,Female,No,Tablet,TBL345,3,Cancelled,Credit Card,3145.64,786.41,4,2024-09-23,Same Day,"Extended Warranty, Extended Warranty",145.55


In [123]:
orders_preview.shape

(12135, 18)

In [124]:
orders.shape

(12135, 13)

In [125]:
orders.info()

<class 'pandas.DataFrame'>
RangeIndex: 12135 entries, 0 to 12134
Data columns (total 13 columns):
 #   Column             Non-Null Count  Dtype         
---  ------             --------------  -----         
 0   order_id           12135 non-null  int64         
 1   Customer ID        12135 non-null  int64         
 2   product_id         12135 non-null  int64         
 3   Rating             12135 non-null  int64         
 4   Order Status       12135 non-null  str           
 5   Payment Method     12135 non-null  str           
 6   Total Price        12135 non-null  float64       
 7   Unit Price         12135 non-null  float64       
 8   Quantity           12135 non-null  int64         
 9   Purchase Date      12135 non-null  datetime64[us]
 10  Shipping Type      12135 non-null  str           
 11  Add-ons Purchased  9196 non-null   str           
 12  Add-on Total       12135 non-null  float64       
dtypes: datetime64[us](1), float64(3), int64(5), str(4)
memory usage: 1.2 MB


## Interpretación

Se ha construido correctamente la tabla **orders**, que actuará como tabla de hechos del modelo relacional.

Cada pedido queda asociado a un cliente mediante `Customer ID` y a un producto mediante `product_id`, permitiendo establecer las relaciones entre las distintas tablas del modelo.

La tabla contiene **12.135 pedidos** y **13 columnas**, incluyendo la clave primaria `order_id`.

Durante la construcción se conservó la información histórica de cada compra, manteniendo atributos propios de la transacción como la fecha de compra, la cantidad, el precio unitario, el precio total y el método de pago.

# Exportación de las tablas

Una vez finalizada la construcción del modelo relacional, se exportan las tablas resultantes a la carpeta `data/processed`.

Estos archivos serán utilizados en la siguiente fase del proyecto para crear e importar la información en la base de datos MySQL.

In [126]:
# ==================================
# Exportación de las tablas
# ==================================

customers.to_csv(
    "../data/processed/customers.csv",
    index=False
)

products.to_csv(
    "../data/processed/products.csv",
    index=False
)

orders.to_csv(
    "../data/processed/orders.csv",
    index=False
)

# Exportación de datos para MySQL

Se renombran las columnas utilizando la convención `snake_case` y se exportan los archivos CSV que serán importados en MySQL.

In [127]:
# ==================================
# Preparación de datos para MySQL
# ==================================

customers_sql = customers.rename(
    columns={
        "Customer ID": "customer_id",
        "Age": "age",
        "Gender": "gender",
        "Loyalty Member": "loyalty_member"
    }
)

customers_sql.head()

,customer_id,age,gender,loyalty_member
0,19033,47,Female,No
1,1402,77,Male,No
2,14264,36,Male,No
3,9976,36,Female,No
4,19471,54,Female,No


In [128]:
products_sql = products.rename(
    columns={
        "product_id": "product_id",
        "SKU": "sku",
        "Product Type": "product_type"
    }
)

products_sql.head()

,product_id,sku,product_type
0,1,HDP456,Headphones
1,2,LTP123,Laptop
2,3,SKU1001,Smartphone
3,4,SKU1002,Tablet
4,5,SKU1003,Smartwatch


In [129]:
orders_sql = orders.rename(
    columns={
        "order_id": "order_id",
        "Customer ID": "customer_id",
        "product_id": "product_id",
        "Rating": "rating",
        "Order Status": "order_status",
        "Payment Method": "payment_method",
        "Total Price": "total_price",
        "Unit Price": "unit_price",
        "Quantity": "quantity",
        "Purchase Date": "purchase_date",
        "Shipping Type": "shipping_type",
        "Add-ons Purchased": "addons_purchased",
        "Add-on Total": "addon_total"
    }
)

orders_sql.head()

,order_id,customer_id,product_id,rating,order_status,payment_method,total_price,unit_price,quantity,purchase_date,shipping_type,addons_purchased,addon_total
0,1,19033,10,4,Completed,Bank Transfer,4718.46,786.41,6,2024-09-23,Standard,"Impulse Item, Impulse Item",187.94
1,2,1402,6,2,Cancelled,Paypal,3955.95,791.19,5,2024-09-23,Overnight,"Accessory,Accessory,Extended Warranty",99.39
2,3,14264,8,2,Completed,Credit Card,11396.80,1139.68,10,2024-09-23,Standard,"Impulse Item, Impulse Item",163.65
3,4,9976,4,3,Completed,Paypal,2223.27,247.03,9,2024-09-23,Overnight,Accessory,8.97
4,5,19471,10,3,Cancelled,Credit Card,3145.64,786.41,4,2024-09-23,Same Day,"Extended Warranty, Extended Warranty",145.55


In [130]:
customers_sql["loyalty_member"] = customers_sql["loyalty_member"].replace({
    "Yes": 1,
    "No": 0
})

In [131]:
customers_sql.to_csv(
    "../data/sql/customers_sql.csv",
    index=False
)

products_sql.to_csv(
    "../data/sql/products_sql.csv",
    index=False
)

orders_sql.to_csv(
    "../data/sql/orders_sql.csv",
    index=False
)

In [133]:
orders_sql.shape

(12135, 13)